In [187]:
import json
def word_to_l_s(word):
    # 1. Break into Absolute Primitives
    # Note: 'ँ' (Chandrabindu) is removed from the weight-contributing matras
    matra_to_vowel = {
        'ा': 'आ', 'ि': 'इ', 'ी': 'ई', 'ु': 'उ', 'ू': 'ऊ', 
        'ृ': 'ऋ', 'े': 'ए', 'ै': 'ऐ', 'ो': 'ओ', 'ौ': 'औ',
        'ं': 'अं', 'ः': 'अः'
    }
    
    primitives = []
    i = 0
    while i < len(word):
        char = word[i]
        next_char = word[i+1] if i+1 < len(word) else None
        
        # IGNORE Chandrabindu: Just skip it
        if char == 'ँ':
            i += 1
            continue

        if '\u0915' <= char <= '\u0939' or '\u0958' <= char <= '\u095f':
            if next_char == '्':
                primitives.append(char + '्')
                i += 2
            elif next_char in matra_to_vowel:
                primitives.append(char + '्')
                primitives.append(matra_to_vowel[next_char])
                i += 2
            else:
                primitives.append(char + '्')
                primitives.append('अ')
                i += 1
        else:
            if char != '्': primitives.append(char)
            i += 1

    # 2. Group Primitives into Syllables
    syllables = []
    current_syl = []
    for p in primitives:
        current_syl.append(p)
        if not p.endswith('्'): # End syllable at the vowel
            syllables.append(current_syl)
            current_syl = []
    
    if current_syl: 
        if syllables:
            syllables[-1].extend(current_syl)
        else:
            syllables.append(current_syl)

    # 3. Rhythmic Weighting Logic
    guru_vowels = {'आ', 'ई', 'ऊ', 'ए', 'ऐ', 'ओ', 'औ', 'अं', 'अः'}
    weights = []
    
    for idx, syl in enumerate(syllables):
        # Identify the vowel in this syllable (default to 'अ' if only half-consonants)
        v = next((p for p in syl if not p.endswith('्')), 'अ')
        
        # Base: Is the vowel naturally heavy?
        is_s = v in guru_vowels
        
        # Rule A: Trailing half-consonant in the SAME syllable (e.g., 'निर्')
        v_idx = syl.index(v) if v in syl else -1
        if not is_s and any(p.endswith('्') for p in syl[v_idx+1:]):
            is_s = True
            
        # Rule B: Cluster Promotion (2+ consonants in the START of the NEXT syllable)
        if not is_s and idx + 1 < len(syllables):
            next_syl = syllables[idx + 1]
            consonant_count = 0
            for p in next_syl:
                if p.endswith('्'):
                    consonant_count += 1
                else:
                    break
            
            if consonant_count >= 2:
                is_s = True
        
        weights.append("S" if is_s else "l")
                
    return "".join(weights)

voc = json.load(open(r"C:\Users\lekhp\OneDrive\Desktop\clean_nepali_words.json", encoding="utf-8"))

voc_prim = {word: word_to_l_s(word) for word in voc}
prim_voc = {r: [w for w, rh in voc_prim.items() if rh == r] for r in set(voc_prim.values())}
    
def predict_words(formula, input_string):
    formula_translation = "".join([voc_prim[x] for x in input_string.split()])
    next_formula = formula[len(formula_translation):]
    next_formula_possible = [next_formula[:x] for x in range(max(4, len(next_formula)+1), 1, -1)]
    next_words = [item for x in next_formula_possible for item in prim_voc.get(x, [])]
    return next_words

print(len(voc))

43880


In [252]:
import random

# Assuming get_weight is the function we wrote earlier
# and rev_voc is your dictionary { "lS": ["मलाई", "कतै"], ... }
def get_weight(word):
    return "".join([voc_prim[x] for x in word.split()])

formula = "lSSlSSlSSlSS"
word_for_now = ""

# The loop continues as long as the weight of our string is shorter than the formula
while len(get_weight(word_for_now)) < len(formula):
    

    new_word = random.choice(predict_words(formula, word_for_now)) 
    word_for_now += " " + new_word 

print(f"Final Poem: {word_for_now}")
print(f"Final Rhythm: {get_weight(word_for_now)}")

Final Poem:  हिरा सूर्य भन्नै अजेन्डा उछिन्ने
Final Rhythm: lSSlSSlSSlSS
